In [ ]:
import jupyter_core.paths
print(jupyter_core.paths.jupyter_runtime_dir())

In [ ]:
import sys
#!pip install polars --target ./my_custom_packages
sys.path.append('./my_custom_packages') 
# Polars, fastexcel, hvplot, "vegafusion[embed]>=1.5.0", ""vl-convert-python>=1.6.0"", dash, dash_bootstrap_components, matplotlib, seaborn, altair

import polars as pl
import pandas as pd

import matplotlib.pyplot as plt

pl.Config.set_tbl_rows(50)      # Display all rows
pl.Config.set_tbl_cols(-1)      # Display all columns

import seaborn as sns

import altair as alt
alt.data_transformers.enable("vegafusion")

print(f"Seaborn version: {sns.__version__}")
print(f"Seaborn file location: {sns.__file__}")


In [ ]:
# May, June, July

loc_quarter = pl.read_parquet("loc_quarter.parquet")
display(loc_quarter.head())

forfait_quarter = pl.read_parquet("forfait_quarter.parquet")
display(forfait_quarter.head())

# Some msisdn are registered to an old site id and that's why they don't appear in the forfait mai data

sites = pl.read_excel("REF_sites_V4.xlsx")
display(sites.head())
#display(sites.shape[0])
#display(sites["nom_site"].unique().count())


sig_cell = pl.read_parquet("rf_sig_cell_v3.parquet")
display(sig_cell.head())


In [ ]:
forfait_quarter = forfait_quarter.with_columns(
    pl.col("msisdn").cast(pl.String),
    pl.col("CA").cast(pl.Float64, strict = False)
)
#display(forfait_quarter.head())


display("Number of connections to cells:")
display(loc_quarter.shape[0])
display("Number of individuals that connected to a cell:")
display(loc_quarter["msisdn"].n_unique())
display("Number of individuals that bought a mobile plan without engagement:")
display(forfait_quarter["msisdn"].n_unique())


# Both the total row count and the unique user count are exactly 4 559 462
# Which implies that the dataset was pre filtered to keep only the most frequented site of each user.

In [ ]:
display("Number of sites in madagascar:")
display(sites.shape[0])
display("Number of cells in madagascar (23 per site on average):")
display(sig_cell.shape[0])

partial_loc_info = sites.join(sig_cell, left_on = "code_site GMT", right_on = "sig_code_site", how = "left")

display("Number of cells that linked to a site:")
display(partial_loc_info.shape[0])
# About a thousand cells are not linked to a site: 3% of lost cell info

full_loc_info = loc_quarter.join(partial_loc_info, left_on = "site_id", right_on = "sig_lac_ci", how = "inner")

display("Number of user to cell connection that found site info:")
display(full_loc_info.shape[0])
# About seventy thousand user's cell connection were not found in our loc dataset: 1% of lost user loc info


In [ ]:
df_merged = full_loc_info.join(forfait_quarter, on = "msisdn", how = "inner")

display("Data with a cell location and at least one mobile plan purchase with no engagement.")
display("Number of mobile plan sales without engagement during the quarter:")
display(df_merged.shape[0])

display("Number of individuals that bought a mobile plan without engagement during the quarter:")
display(df_merged["msisdn"].unique().count())

#display(df_merged.columns)

df_merged = df_merged.with_columns([
    pl.col("x").cast(pl.Float64, strict = False),
    pl.col("y").cast(pl.Float64, strict = False),
    pl.col("date").cast(pl.Date, strict = False)
])

cols_to_keep = [
    'msisdn',
    'site_id',
    'nb_jours',
    

    'date',
    'Gamme_groupe',
    'Nom du forfait',
    'CA',
    'Group_canal',
    
    'Typologie',
    'Typologie site', 
    'sig_region_name',
    'sig_district_name',
    'sig_commune_name',
    'x', 
    'y',
    "nom_site",
    
    'Max_RAT'
]

df_merged = df_merged.select(cols_to_keep)
display(df_merged.head())


In [ ]:
df_merged = df_merged.filter(pl.col("Group_canal").is_not_null()) 

display(
    forfait_quarter.group_by("Group_canal").len().with_columns(
        (((pl.col("len")/pl.col("len").sum()) * 100).round(2)).alias("Prop in %")
    ).sort("Prop in %", descending = True)
)

display(
    forfait_quarter.group_by('Nom du forfait').len().sort('Nom du forfait').with_columns(
        (((pl.col("len")/pl.col("len").sum()) * 100).round(2)).alias("Prop in %")
    ).sort("Prop in %", descending = True)
)
display(
    forfait_quarter.group_by("Gamme_groupe").len().with_columns(
        (((pl.col("len")/pl.col("len").sum()) * 100).round(2)).alias("Prop in %")
    ).sort("Prop in %", descending = True )
)


df_merged = df_merged.filter(
    pl.col("Group_canal").is_not_null()

) 



In [ ]:
# How many purchase per individual?

display(df_merged.group_by("msisdn").len().select("len").describe())
display(df_merged.group_by("msisdn").len().sort("len", descending=True).head(10))

In [ ]:
# Graphical analyss


In [ ]:
# How much do most individuals spend on a data plan?


plt.hist(df_merged.filter(~df_merged["CA"].is_null())["CA"], color = "blue", edgecolor = "black", bins = 20000)
plt.xlabel("Ca values")
plt.ylabel("Frequency")
plt.title("Boxplot of CA values")
plt.xlim(right = 2000, left = 0)
plt.show()

df_merged["CA"].describe()

In [ ]:
%%time

# Where in madagacar are individual connecting most frequently to internet and with which data technology?


hb = plt.hexbin(
    df_merged["x"], 
    df_merged["y"], 
    gridsize = 50, 
    cmap = 'YlOrRd', 
    mincnt = 1, 
    bins = "log", 
    edgecolors = "white"
)

plt.xticks([])

cb = plt.colorbar(hb, label = "Count (log scale)")

plt.xlabel("X Coor")
plt.ylabel("Y Coor")
plt.title("Individual connection density plot of Madagascar where each hexagone = ~11 kms wide")

plt.show()


# Madagascar is about 550 kilometers wide. 
# At gridsize = 50, each hexagon has an area of about 11 to 14 kilometers wide.

In [ ]:
hb = plt.hexbin(
    df_merged["x"], 
    df_merged["y"], 
    gridsize = 100, 
    cmap = 'YlOrRd', 
    mincnt = 1, 
    bins = "log", 
    edgecolors = "white",
)

plt.xticks([])

cb = plt.colorbar(hb, label = "Count (log scale)")

plt.xlabel("X Coor")
plt.ylabel("Y Coor")
plt.title("Individual connection density plot of Madagascar where each hexagone = ~5 kms wide")
plt.show()

plt.show()

# Madagascar is about 550 kilometers wide. 
# At gridsize = 100, each hexagon has an area of about 5 to 7 kilometers wide.

In [ ]:
%%time

df_bubbles = df_merged.group_by(["x", "y", "Max_RAT"]).len()
#display(df_bubbles.head)


order = ["2G", "3G", "4G_FDD", "4G_TDD", "4G+_FDD", "4G+_TDD", "5G"][::-1]

colors = {
    "2G": "yellow",
    "3G": "orange",
    "4G_FDD": "purple",
    "4G_TDD": "purple",
    "4G+_FDD": "green",
    "4G+_TDD": "green",
    "5G": "blue"
}



sns.scatterplot(
    data = df_bubbles,
    x = "x",
    y = "y",
    hue = "Max_RAT",
    hue_order = order,
    size = "len",
    sizes = (20, 1000),
    palette = colors,
    alpha = 0.4
)

plt.legend(bbox_to_anchor=(1.05, 1))

plt.show()


In [ ]:
# relplot creates subplots based on col
g = sns.relplot(
    data = df_bubbles,
    x = "x",
    y = "y",
    col = "Max_RAT",       
    col_wrap = 3,          # 3 maps per row
    hue = "Max_RAT",
    size = "len",
    sizes = (20, 1000),
    hue_order = order,
    palette = colors,
    alpha = 0.4,
    height = 6            # Height of each subplot
)

g.set_titles("{col_name} Coverage")
g.set_axis_labels("x", "y")
g.fig.suptitle('Distribution by data technology', y=1.05, fontsize=16)

plt.show()

In [ ]:
# Are rural customers buying data differently than urban customers?


df_plot = df_merged.group_by(["Typologie", "Group_canal"]).agg(
        pl.col("CA").sum().alias("Total_CA")
    )

#display(df_plot.head())

stacked_box = df_plot.plot.bar(
    x = alt.X("Typologie", title = "Site"), 
    y = alt.Y("Total_CA", title="Total CA"), 
    color = alt.Color("Group_canal", title="Canal")
)

stacked_box = stacked_box.properties(
    width = 800,
    height = 400,
    title = "Total CA by site and canal"
)

display(stacked_box)

display(df_merged.group_by("Typologie").agg(
    pl.col("CA").mean().alias("average_ca"),
    pl.col("nb_jours").mean().alias("average_nbj"),
    pl.col("msisdn").count().alias("n")
).sort("average_ca", descending=True))

In [ ]:
# Does a better connection drive more spending?


df_sample = df_merged.sample(n = 100000, seed = 4).to_pandas()

sns.violinplot(
    data = df_sample, 
    x = "Max_RAT", 
    y = "CA", 
    order = order,
    cut = 0, # no negative density
    log_scale = True

#    density_norm = "count" # Scale to the counts of that category
)
plt.xlabel("Data technology")
plt.ylabel("CA (log scale)")
plt.title("Violon plot of individual user CA by data technologoy")
plt.show()


counts_df = df_merged.group_by("Max_RAT").len().to_pandas()

sns.barplot(
    counts_df, 
    x = "Max_RAT",
    y = "len",
    order = order
)
plt.xlabel("Data technology")
plt.ylabel("Count")
plt.title("Count plot of users for different data technologoy")

plt.show()

display(
    df_merged.group_by("Max_RAT").agg(
        pl.col("CA").sum().alias("All CA"), 
        pl.col("CA").mean().alias("Average CA"),
        pl.col("msisdn").unique().count().alias("n unique users"), 
        pl.col("msisdn").count().alias("n transaction")

    ).sort("All CA", descending = True)
)


In [ ]:
%%time

# When are users connected the most?

df_ca = df_merged.group_by(["date"]).agg(
    pl.col("CA").sum().alias("Sum_CA")
).with_columns(
    wd = pl.col("date").dt.weekday(),
    weekday = pl.col("date").dt.strftime("%a"),
    week_n = pl.col("date").dt.week()
).sort("date")

pdf = df_ca.to_pandas()

sns.lineplot(
    data = pdf, 
    x = "date", 
    y = "Sum_CA",
    marker = "o"
)

for x, y, day in zip(pdf["date"], pdf["Sum_CA"], pdf["weekday"]):
    plt.text(x, y, day, fontsize = 8, ha = "center", va = "bottom")

plt.xticks(rotation=45)
plt.xlabel("Day")
plt.ylabel("CA")
plt.title("Ca by day for the month of may")

plt.axhline(
    pdf["Sum_CA"].mean(),
    color = "red",
    linestyle = "--",
    label = "Average"
)
plt.legend()

plt.show()


sns.lineplot(
    data = pdf, 
    x = "weekday", 
    y = "Sum_CA",
    marker = "o"
)
plt.xticks(rotation=45)
plt.xlabel("Weekday")
plt.ylabel("CA")
plt.title("Ca by day of the week for the month of may")

plt.show()

pdf.boxplot(column = "Sum_CA", by = "week_n", color = "blue", grid = False)
plt.xlabel("Week")
plt.ylabel("Ca values")
plt.title("Boxplot of CA values by Week")

plt.show()





In [ ]:
# Do most connected individual spend the most?


df_tca_nbj_sample = df_merged.group_by("msisdn").agg(
    pl.col("CA").sum().alias("Total_CA"), pl.col("nb_jours").unique().first().alias("nb_jours")
)

#display(df_tca_nbj_sample.head())

corr_matrix = df_tca_nbj_sample.select(
    pl.corr("Total_CA", "nb_jours").alias("correlation_nbj_TCA")
)
display(corr_matrix)

sns.scatterplot(
    data = df_tca_nbj_sample, 
    x = "nb_jours", 
    y = "Total_CA",
    alpha = 0.4
)
plt.xlabel("Number of days with established cell connection in a month")
plt.ylabel("Total CA by user")
plt.title("User CA value by numbers of days connected")

plt.show()

In [ ]:
# Saving dataset for further statistical analysis

df_merged.write_parquet("df_merged.parquet")
